In [1]:
from pyspark import SparkContext
from pyspark.sql import SparkSession

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
BASE_PATH = "/content/drive/MyDrive/ds200/assignment-4/"

customer_list = "Customer_List.csv"
order_items = "Order_Items.csv"
order_reviews = "Order_Reviews.csv"
orders = "Orders.csv"
products = "Products.csv"

In [4]:
sc = SparkContext.getOrCreate()

spark = SparkSession.builder.appName("LAB4SparkDataFrame").getOrCreate()

In [5]:
display(spark)

# Đề bài

Fecom Inc. là công ty thương mại điện tử có trụ sở tại Berlin, Đức. Từ năm 2022 đến 2024, công ty đã ghi nhận 99.441 đơn hàng từ 102.727 khách hàng duy nhất và theo dõi giao dịch của 3.095 người bán. Bộ dữ liệu chứa thông tin về:

- Đơn hàng (Orders): Thông tin về trạng thái đơn hàng, thời gian mua, duyệt, giao hàng...
- Khách hàng (Customer_List): Thông tin về ngày đăng ký, ngày đặt hàng đầu tiên, địa chỉ, độ tuổi, giới tính...
- Chi tiết đơn hàng (Order_Items): Danh sách sản phẩm, giá, phí vận chuyển, ngày giao hàng dự kiến...
- Sản phẩm (Products): Thông tin về danh mục, kích thước, trọng lượng sản phẩm...
- Đánh giá đơn hàng (Order_Reviews): Điểm đánh giá, tiêu đề và nội dung bình luận, thời gian đánh giá...

Dữ liệu này đến từ 338 thành phố tại 28 quốc gia, với 32.951 sản phẩm thuộc 72 danh mục khác nhau. Mục tiêu của bài thực hành là sử dụng Spark DataFrame để thực hiện các phân tích bán hàng và tiếp thị.

# Bài làm

## 1.	Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột

In [6]:
df_customers = spark.read.option("header","true").option("sep",";")\
      .option("inferSchema","true")\
      .csv(BASE_PATH + customer_list)

In [7]:
df_customers.show(10)

+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|     Customer_Trx_ID|       Subscriber_ID|Subscribe_Date|First_Order_Date|Customer_Postal_Code|Customer_City|Customer_Country|Customer_Country_Code|Age|Gender|
+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|1e959e1f5920cba43...|9765e039028279fd2...|    2023-07-08|      2023-07-09|            FR-75005|        Paris|          France|                   FR| 29|  Male|
|9877437582f263da7...|a75e134e7eb6f96e2...|    2024-03-23|      2024-04-11|           PL-00-001|       Warsaw|          Poland|                   PL| 38|  Male|
|fa6fbbb2080646aca...|2fdac27295500e820...|    2023-05-12|      2023-06-01|             NL-1012|    Amsterdam|     Netherlands|                   NL| 35|Female|
|a4c9ff14ae7620126...|e9ab8fd8ea96

In [8]:
df_customers.printSchema()

root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)



In [9]:
df_order_items = spark.read.option("header","true").option("sep",";")\
      .option("inferSchema","true")\
      .csv(BASE_PATH + order_items)
df_order_reviews = spark.read.option("header","true").option("sep",";")\
      .option("inferSchema","true")\
      .csv(BASE_PATH + order_reviews)
df_orders = spark.read.option("header","true").option("sep",";")\
      .option("inferSchema","true")\
      .csv(BASE_PATH + orders)
df_products = spark.read.option("header","true").option("sep",";")\
      .option("inferSchema","true")\
      .csv(BASE_PATH + products)

## 2.	Thống kê tổng số đơn hàng, số lượng khách hàng và người bán.

## Tổng số đơn hàng

In [10]:
count_orders = df_orders.count()
print(f"Tổng số đơn hàng: {count_orders}")

Tổng số đơn hàng: 99441


## Số lượng khách hàng

In [11]:
count_customers = df_customers.count()
print(f"Tổng số khách hàng: {count_customers}")

Tổng số khách hàng: 102727


In [12]:
df_order_items.describe().show()

+-------+--------------------+------------------+--------------------+--------------------+-----------------+------------------+
|summary|            Order_ID|     Order_Item_ID|          Product_ID|           Seller_ID|            Price|     Freight_Value|
+-------+--------------------+------------------+--------------------+--------------------+-----------------+------------------+
|  count|              112650|            112650|              112650|              112650|           112650|            112650|
|   mean|                NULL|1.1978339991122948|                NULL|                NULL|120.6537390147135| 19.99031992898376|
| stddev|                NULL|0.7051240313951719|                NULL|                NULL|183.6339280502596|15.806405412297115|
|    min|00010242fe8c5a6d1...|                 1|00066f42aeeb9f300...|0015a82c2db000af6...|             0.85|               0.0|
|    max|fffe41c64501cc87c...|                21|fffe9eeff12fcbd74...|ffff564a4f9085cd2...|      

In [13]:
count_sellers = df_order_items.select("seller_id").distinct().count()
print(f"Tổng số người bán: {count_sellers}")

Tổng số người bán: 3095


## 3.	Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần.

In [14]:
df_customers.limit(10).show()

+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|     Customer_Trx_ID|       Subscriber_ID|Subscribe_Date|First_Order_Date|Customer_Postal_Code|Customer_City|Customer_Country|Customer_Country_Code|Age|Gender|
+--------------------+--------------------+--------------+----------------+--------------------+-------------+----------------+---------------------+---+------+
|1e959e1f5920cba43...|9765e039028279fd2...|    2023-07-08|      2023-07-09|            FR-75005|        Paris|          France|                   FR| 29|  Male|
|9877437582f263da7...|a75e134e7eb6f96e2...|    2024-03-23|      2024-04-11|           PL-00-001|       Warsaw|          Poland|                   PL| 38|  Male|
|fa6fbbb2080646aca...|2fdac27295500e820...|    2023-05-12|      2023-06-01|             NL-1012|    Amsterdam|     Netherlands|                   NL| 35|Female|
|a4c9ff14ae7620126...|e9ab8fd8ea96

In [15]:
df_orders.limit(10).show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            Order_ID|     Customer_Trx_ID|Order_Status|Order_Purchase_Timestamp|  Order_Approved_At|Order_Delivered_Carrier_Date|Order_Delivered_Customer_Date|Order_Estimated_Delivery_Date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2023-10-02 10:56:00|2023-10-02 11:07:00|         2023-10-04 19:55:00|          2023-10-10 21:25:00|          2023-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2024-07-24 20:41:00|2024-07-26 03:24:00|         2024-07-26 14:31:00|          2024-08-07 15:27:00|          2024-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [16]:
df_orders_customers = df_orders.join(df_customers, on="Customer_Trx_ID", how = "inner")

In [17]:
df_results_3 = df_orders_customers.groupBy("Customer_Country").count().orderBy("count", ascending=False)

In [18]:
df_results_3.printSchema()

root
 |-- Customer_Country: string (nullable = true)
 |-- count: long (nullable = false)



In [19]:
df_results_3.show()

+----------------+-----+
|Customer_Country|count|
+----------------+-----+
|         Germany|41754|
|          France|12848|
|     Netherlands|11629|
|         Belgium| 5464|
|         Austria| 5043|
|     Switzerland| 3640|
|  United Kingdom| 3382|
|          Poland| 2139|
|         Czechia| 2034|
|           Italy| 2025|
|           Spain| 1651|
|        Portugal| 1336|
|          Sweden|  975|
|         Denmark|  905|
|          Serbia|  746|
|          Norway|  716|
|        Slovakia|  534|
|        Slovenia|  495|
|          Turkey|  485|
|          Greece|  412|
+----------------+-----+
only showing top 20 rows


## 4.	Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm tăng dần, tháng giảm dần)

In [20]:
from pyspark.sql import functions as F

In [21]:
df_orders.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)



In [22]:
df_orders_with_purchase_month_year = df_orders.withColumn("Year_Purchase", F.year("Order_Purchase_Timestamp"))\
                                              .withColumn("Month_Purchase", F.month("Order_Purchase_Timestamp"))

In [23]:
df_results_4 = df_orders_with_purchase_month_year.groupBy("Year_Purchase","Month_Purchase")\
                .agg(F.count("Order_ID").alias("Total Orders"))\
                .orderBy(F.col("Year_Purchase").asc(), F.col("Month_Purchase").desc())

In [24]:
df_results_4.show()

+-------------+--------------+------------+
|Year_Purchase|Month_Purchase|Total Orders|
+-------------+--------------+------------+
|         2022|            12|           1|
|         2022|            10|         324|
|         2022|             9|           4|
|         2023|            12|        5673|
|         2023|            11|        7544|
|         2023|            10|        4631|
|         2023|             9|        4285|
|         2023|             8|        4331|
|         2023|             7|        4026|
|         2023|             6|        3245|
|         2023|             5|        3700|
|         2023|             4|        2404|
|         2023|             3|        2682|
|         2023|             2|        1780|
|         2023|             1|         800|
|         2024|            10|           4|
|         2024|             9|          16|
|         2024|             8|        6512|
|         2024|             7|        6292|
|         2024|             6|  

## 5.	Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến 5).
Lưu ý: Cần xử lý các giá trị ngoại lệ và NULL trong cột Review_Score

In [25]:
df_order_reviews.describe().show()

+-------+--------------------+--------------------+------------------+-----------------------+-------------------------+--------------------+
|summary|           Review_ID|            Order_ID|      Review_Score|Review_Comment_Title_En|Review_Comment_Message_En|Review_Creation_Date|
+-------+--------------------+--------------------+------------------+-----------------------+-------------------------+--------------------+
|  count|               99270|               99225|             99225|                  11543|                    40878|               99221|
|   mean|                NULL|                NULL|4.0864214950162765|      8950567.848387096|                   7.5625|                NULL|
| stddev|                NULL|                NULL|1.3475858938971887|   1.5193692881366396E8|        3.518957873593975|                NULL|
|    min|0001239bc1de2e33c...|00010242fe8c5a6d1...|                 1|                5 stars|      Arrived quickly ...| I can't understa...|
|    m

In [26]:
df_order_reviews = df_order_reviews.dropna(how='all')

In [27]:
df_order_reviews.filter(F.col("Review_Score").isNull()).count()

45

In [28]:
df_order_reviews = df_order_reviews.dropna(subset=["Review_Score"])

In [29]:
df_order_reviews.filter(F.col("Review_Score").isNull()).count()

0

In [30]:
df_order_reviews.select("Review_Score").distinct().show()

+----------------+
|    Review_Score|
+----------------+
|               3|
|               5|
|2024-07-05 11:11|
|               1|
|               4|
|               2|
|2024-04-07 04:19|
+----------------+



In [31]:
valid_scores = ["1", "2", "3", "4", "5"]
df_order_reviews = df_order_reviews.filter(F.col("Review_Score").isin(valid_scores))

In [32]:
df_order_reviews.select("Review_Score").distinct().show()

+------------+
|Review_Score|
+------------+
|           3|
|           5|
|           1|
|           4|
|           2|
+------------+



In [33]:
df_order_reviews = df_order_reviews.withColumn("Review_Score", F.col("Review_Score").cast("int"))

In [34]:
df_results_5 = df_order_reviews.groupBy("Review_Score")\
              .agg(F.avg("Review_Score").alias("Average Score"),
                   F.count("Review_Score").alias("Total Reviews"))

In [35]:
df_results_5.show()

+------------+-------------+-------------+
|Review_Score|Average Score|Total Reviews|
+------------+-------------+-------------+
|           1|          1.0|        11424|
|           3|          3.0|         8179|
|           5|          5.0|        57328|
|           4|          4.0|        19141|
|           2|          2.0|         3151|
+------------+-------------+-------------+



## 6.  Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm

In [38]:
df_orders.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)



In [39]:
df_orders_2024 = df_orders.filter(F.year("Order_Purchase_Timestamp") == 2024)

In [44]:
df_joined = df_orders_2024.join(df_order_items, df_orders_2024.Order_ID == df_order_items.Order_ID) \
                          .join(df_products, "Product_ID")

In [45]:
df_revenue = df_joined.withColumn(
    "Item_Revenue",
    F.col("Price").cast("double") + F.col("Freight_Value").cast("double")
)

In [48]:
df_results_6 = df_revenue.groupBy("Product_Category_Name") \
    .agg(F.round(F.sum("Item_Revenue"), 2).alias("Total_Revenue")) \
    .orderBy(F.col("Total_Revenue").desc())

In [49]:
df_results_6.show()

+---------------------+-------------+
|Product_Category_Name|Total_Revenue|
+---------------------+-------------+
|        Health_Beauty|    885191.12|
|        Watches_Gifts|    771986.75|
|       Bed_Bath_Table|     650794.7|
|       Sports_Leisure|    621999.34|
| Computers_Accesso...|    594771.04|
|           Housewares|    491576.96|
|      Furniture_Decor|    476466.13|
|                 Auto|    404210.57|
|                 Baby|    299052.56|
|           Cool_Stuff|    273910.05|
|         Garden_Tools|    259068.32|
|            Telephony|    217452.13|
|            Perfumery|    204562.54|
|                 Toys|    200634.07|
|     Office_Furniture|    181745.73|
|           Stationery|    164743.85|
|             Pet_Shop|    152804.94|
| Construction_Tool...|    141187.34|
|          Electronics|    134265.45|
|  Musical_Instruments|    121476.31|
+---------------------+-------------+
only showing top 20 rows


## 9.  Nhóm khách hàng dựa trên số lượng đơn hàng, giá trị trung bình của đơn hàng và tần suất mua sắm.

In [47]:
df_orders_customers_order_items = df_orders.join(df_customers, on="Customer_Trx_ID", how = "inner")\
                                    .join(df_order_items, on="Order_ID", how = "inner")

In [51]:
df_orders_customers_order_items.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Order_Item_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Seller_ID: string (nullable = true)
 |-- Shipping_Limit_Date: 

In [52]:
df_grouped = df_orders_customers_order_items.groupBy("Customer_Trx_ID","Order_ID").agg(F.sum("Price").alias("Total_Price"))

In [54]:
df_grouped.show()

+--------------------+--------------------+-----------+
|     Customer_Trx_ID|            Order_ID|Total_Price|
+--------------------+--------------------+-----------+
|228da462edabbbc68...|65312c6e4ce03c0fb...|      54.99|
|d19f254a07da73a05...|90349f264a3d6a252...|      270.0|
|7203eb38fea2b4a15...|5869074071e0f56e9...|      300.0|
|bf22881f1f23a7991...|813f8916cd12b887b...|       27.9|
|973e21d9b220229ae...|c432ba5b3bb83966c...|       25.0|
|cf801016b25bee769...|965c963c874eedc47...|       89.9|
|4ad5a269a2d59d2c8...|b323a1bf3949211f3...|      64.99|
|640246e27e1df1ffe...|cc06813c8d9909633...|      119.0|
|59cf1a479d104e385...|bcdbca456c84595f3...|       34.9|
|21e121469a31f5b4e...|44fdca9e41cfd8248...|      132.2|
|df059d8e5672ffde0...|939e85580cc5404ab...|       39.6|
|2680dc46f78c878b1...|1ac4c8f4bb93ff8b5...|      49.91|
|855f0b8c477c57d9c...|9d43eaf1f72a6d200...|      140.0|
|943a02b9d5564146f...|afde85ca0156001aa...|      380.0|
|a7dbd3fd213f08708...|0e2f2a11ed7233e53...|     

In [55]:
df_grouped = df_grouped.groupBy("Customer_Trx_ID").agg(
    F.count("Order_ID").alias("Total_Orders"),
    F.avg("Total_Price").alias("Average_Order_Value"))

In [57]:
df_grouped.orderBy("Total_Orders", ascending=False).show()

+--------------------+------------+-------------------+
|     Customer_Trx_ID|Total_Orders|Average_Order_Value|
+--------------------+------------+-------------------+
|0721e1c4b91bc6ded...|           1|               89.9|
|cf32c953e3f9c1fe1...|           1|               72.9|
|36a1aa63bf2ebcd49...|           1|              86.99|
|7a399cda317ad8719...|           1|               29.9|
|9a5d3906304996948...|           1|              224.1|
|fd667d6a6784c49bf...|           1|              168.0|
|80447be02d8f3f3f3...|           1|               25.9|
|c90e1fc62717b61b9...|           1|              56.99|
|088935be5a63fd416...|           1|              59.99|
|d83a7e09753bfb23e...|           1|              143.8|
|0b60138b038f460ba...|           1|               65.0|
|7efa4ffd3323a20dd...|           1|             149.99|
|6e48d7670ed33717b...|           1|             116.99|
|2f0110b33a975d9e2...|           1|               23.7|
|82c91e8e62d9d79bf...|           1|             

## 10. Xếp hạng các seller dựa trên tổng doanh thu và số lượng đơn hàng bán được.

In [58]:
df_grouped = df_orders_customers_order_items.groupBy("seller_id").agg(
    F.count("Order_ID").alias("Total_Orders"),
    F.sum("Price").alias("Total_Revenue"))

In [60]:
df_grouped.orderBy(["Total_Orders","Total_Revenue"],ascending=False).show()

+--------------------+------------+------------------+
|           seller_id|Total_Orders|     Total_Revenue|
+--------------------+------------+------------------+
|6560211a19b47992c...|        2033|         123304.83|
|4a3ca9315b744ce9f...|        1987|200472.91999999818|
|1f50f920176fa81da...|        1931| 106939.2100000013|
|cc419e0650a3c5ba7...|        1775| 104288.4199999996|
|da8622b14eb17ae28...|        1551|160236.56999999937|
|955fee9216a65b617...|        1499|135171.69999999998|
|1025f0e2d44d7041d...|        1428|138968.55000000016|
|7c67e1448b00f6e96...|        1364|187923.89000000124|
|ea8482cd71df3c196...|        1203| 37177.52000000018|
|7a67c85e85bb2ce85...|        1171| 141745.5300000006|
|4869f7a5dfa277a7d...|        1156|229472.62999999907|
|3d871de0142ce09b7...|        1147| 94914.20000000022|
|8b321bb669392f516...|        1018|17535.689999999904|
|cca3071e3e9bb7d12...|         830| 64009.89000000029|
|620c87c171fb2a6dd...|         798|114774.50000000032|
|a1043bafd